# Lesson 14 — Testing LangGraph Agents

## What you will learn
- **Unit testing** individual nodes without running the full graph
- **Integration testing** the compiled graph with mock LLMs
- **Snapshot testing** — assert exact state after each node
- **Routing tests** — verify conditional edges go to correct nodes
- How to use `pytest` with LangGraph

## Testing pyramid for LangGraph
```
        [E2E tests]          ← full graph + real LLM (slow, expensive)
      [Integration tests]    ← compiled graph + mock LLM  ← focus here
    [Unit tests]             ← individual node functions   ← fast & cheap
```

> **Key insight:** Nodes are plain Python functions — they're trivially unit-testable.
> Mock the LLM at the node level; test routing logic separately.

In [ ]:
# !pip install langgraph pytest

## Step 1 — The Agent Under Test

We'll test a simple sentiment-routing agent from Lesson 2.

In [ ]:
from typing import TypedDict, Literal, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

class ReviewState(TypedDict):
    review: str
    sentiment: str
    response: str

def classify_node(state: ReviewState) -> dict:
    text = state["review"].lower()
    if any(w in text for w in ["great", "love", "excellent", "amazing"]):
        sentiment = "positive"
    elif any(w in text for w in ["bad", "terrible", "hate", "awful"]):
        sentiment = "negative"
    else:
        sentiment = "neutral"
    return {"sentiment": sentiment}

def handle_positive(state: ReviewState) -> dict:
    return {"response": "Thank you! We're thrilled you had a great experience!"}

def handle_negative(state: ReviewState) -> dict:
    return {"response": "We're sorry to hear that. We'll work hard to improve!"}

def handle_neutral(state: ReviewState) -> dict:
    return {"response": "Thank you for your feedback!"}

def route_by_sentiment(state: ReviewState) -> Literal["handle_positive", "handle_negative", "handle_neutral"]:
    return f"handle_{state['sentiment']}"

builder = StateGraph(ReviewState)
builder.add_node("classify",       classify_node)
builder.add_node("handle_positive", handle_positive)
builder.add_node("handle_negative", handle_negative)
builder.add_node("handle_neutral",  handle_neutral)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route_by_sentiment, {
    "handle_positive": "handle_positive",
    "handle_negative": "handle_negative",
    "handle_neutral":  "handle_neutral",
})
builder.add_edge("handle_positive", END)
builder.add_edge("handle_negative", END)
builder.add_edge("handle_neutral",  END)
graph = builder.compile()

print("Agent compiled!")

## Step 2 — Unit Testing Nodes

Nodes are plain functions → call them directly with a state dict.

In [ ]:
def test_classify_positive():
    state = {"review": "This is amazing!", "sentiment": "", "response": ""}
    result = classify_node(state)
    assert result["sentiment"] == "positive", f"Expected positive, got {result['sentiment']}"
    print("✅ test_classify_positive passed")

def test_classify_negative():
    state = {"review": "Terrible product, I hate it.", "sentiment": "", "response": ""}
    result = classify_node(state)
    assert result["sentiment"] == "negative"
    print("✅ test_classify_negative passed")

def test_classify_neutral():
    state = {"review": "It was okay.", "sentiment": "", "response": ""}
    result = classify_node(state)
    assert result["sentiment"] == "neutral"
    print("✅ test_classify_neutral passed")

def test_handle_positive_returns_only_changed_key():
    state = {"review": "Great!", "sentiment": "positive", "response": ""}
    result = handle_positive(state)
    # Node should return ONLY the keys it changes
    assert "response" in result
    assert "sentiment" not in result, "Nodes must NOT return unchanged keys"
    print("✅ test_handle_positive_returns_only_changed_key passed")

test_classify_positive()
test_classify_negative()
test_classify_neutral()
test_handle_positive_returns_only_changed_key()

## Step 3 — Routing Tests

Test the routing function independently — no graph execution needed.

In [ ]:
def test_routing():
    cases = [
        ({"review": "", "sentiment": "positive", "response": ""}, "handle_positive"),
        ({"review": "", "sentiment": "negative", "response": ""}, "handle_negative"),
        ({"review": "", "sentiment": "neutral",  "response": ""}, "handle_neutral"),
    ]
    for state, expected in cases:
        result = route_by_sentiment(state)
        assert result == expected, f"Expected {expected}, got {result}"
    print("✅ test_routing passed — all 3 routes correct")

test_routing()

## Step 4 — Integration Tests (full graph invoke)

In [ ]:
def test_full_graph_positive():
    result = graph.invoke({"review": "I love this amazing product!", "sentiment": "", "response": ""})
    assert result["sentiment"] == "positive"
    assert "thrilled" in result["response"].lower()
    print("✅ test_full_graph_positive passed")

def test_full_graph_negative():
    result = graph.invoke({"review": "Terrible, absolutely hate it.", "sentiment": "", "response": ""})
    assert result["sentiment"] == "negative"
    assert "sorry" in result["response"].lower()
    print("✅ test_full_graph_negative passed")

def test_full_graph_preserves_input():
    review = "Decent product overall."
    result = graph.invoke({"review": review, "sentiment": "", "response": ""})
    assert result["review"] == review, "Input keys must be preserved in final state"
    print("✅ test_full_graph_preserves_input passed")

test_full_graph_positive()
test_full_graph_negative()
test_full_graph_preserves_input()

## Step 5 — Snapshot Testing with get_state()

Use `MemorySaver` + streaming to capture state at each node.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
graph_with_memory = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "test-snapshot-001"}}
result = graph_with_memory.invoke(
    {"review": "Amazing experience!", "sentiment": "", "response": ""},
    config=config,
)

# Snapshot all checkpoints
print("State history (snapshots):")
for snap in graph_with_memory.get_state_history(config):
    print(f"  step={snap.values.get('sentiment', '?'):10s} | next={snap.next}")

# Assert final state
final = graph_with_memory.get_state(config)
assert final.values["sentiment"] == "positive"
print("\n✅ Snapshot test passed — sentiment=positive at final checkpoint")

## Step 6 — Running with pytest

Save tests to a file and run with `pytest`.

In [ ]:
test_code = '''
# tests/test_sentiment_agent.py
# Run: pytest tests/test_sentiment_agent.py -v

import pytest
from lesson_14_testing.lesson_14_testing import (
    classify_node, route_by_sentiment, graph
)

@pytest.mark.parametrize("review,expected", [
    ("I love this!", "positive"),
    ("Terrible experience.", "negative"),
    ("It was okay.", "neutral"),
])
def test_classify_node(review, expected):
    result = classify_node({"review": review, "sentiment": "", "response": ""})
    assert result["sentiment"] == expected

def test_full_pipeline():
    result = graph.invoke({"review": "Amazing!", "sentiment": "", "response": ""})
    assert result["sentiment"] == "positive"
    assert result["response"] != ""
'''
print(test_code)

## Key Takeaways

| Test type | How | Speed |
|-----------|-----|-------|
| Unit test node | Call function directly with dict | Fast |
| Unit test routing | Call routing function with dict | Fast |
| Integration test | `graph.invoke(initial_state)` | Medium |
| Snapshot test | `graph.get_state_history(config)` | Medium |
| E2E test | Real LLM + full graph | Slow/expensive |

**Anti-patterns to avoid:**
- Testing nodes that call a real LLM in unit tests (mock instead)
- Not testing routing functions separately
- Forgetting to assert that nodes return ONLY changed keys

## 🏋️ Exercise
1. Add a `very_positive` sentiment path for words like "incredible", "outstanding"
2. Write parametrized pytest tests covering all 4 sentiment paths
3. Write a test that verifies `classify_node` only returns the `sentiment` key (not `review` or `response`)

In [ ]:
# Your exercise solution here
